# RFSoC Fast Overlay Test

Minimal notebook for quickly testing newly generated RFSoC/MTS overlays.

In [1]:
from pathlib import Path
import time
import numpy as np
import matplotlib.pyplot as plt

from pynq import MMIO
from rfsoc_mts import mtsOverlay

try:
    import xrfclk
except Exception as e:
    xrfclk = None
    print('xrfclk import failed:', e)

def status(msg):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}')

In [2]:
status('Loading overlay')
# Change these for each generated overlay.
BITFILE = 'thesis_v16.bit'
ol = mtsOverlay(BITFILE)
EXPECTED_DAC_SR = 9.8304e9
EXPECTED_ADC_SR = 4.0e9
DAC_SR = EXPECTED_DAC_SR
ADC_SR = EXPECTED_ADC_SR
BUF_LEN = int(ol.dac_player.shape[0])

status('Overlay loaded')
print('dac_player dtype:', ol.dac_player.dtype)
print('dac_player shape:', ol.dac_player.shape)
print('BUF_LEN:', BUF_LEN)

[15:03:25] Loading overlay


[15:03:34] Overlay loaded
dac_player dtype: int16
dac_player shape: (131072,)
BUF_LEN: 131072


In [3]:
print(ol.dac_player.shape)
print(ol.ip_dict)
print(ol.mem_dict)

(131072,)
{'clocktreeMTS/MTSclkwiz': {'type': 'xilinx.com:ip:clk_wiz:6.0', 'mem_id': 's_axi_lite', 'memtype': 'REGISTER', 'gpio': {}, 'interrupts': {}, 'parameters': {'C_CLKOUT2_USED': '1', 'C_USER_CLK_FREQ0': '100.0', 'C_AUTO_PRIMITIVE': 'MMCM', 'C_USER_CLK_FREQ1': '100.0', 'C_USER_CLK_FREQ2': '100.0', 'C_USER_CLK_FREQ3': '100.0', 'C_ENABLE_CLOCK_MONITOR': '1', 'C_ENABLE_USER_CLOCK0': '0', 'C_ENABLE_USER_CLOCK1': '0', 'C_ENABLE_USER_CLOCK2': '0', 'C_ENABLE_USER_CLOCK3': '0', 'C_Enable_PLL0': '0', 'C_Enable_PLL1': '0', 'C_REF_CLK_FREQ': '100.0', 'C_PRECISION': '1', 'C_CLKOUT3_USED': '0', 'C_CLKOUT4_USED': '0', 'C_CLKOUT5_USED': '0', 'C_CLKOUT6_USED': '0', 'C_CLKOUT7_USED': '0', 'C_USE_CLKOUT1_BAR': '0', 'C_USE_CLKOUT2_BAR': '0', 'C_USE_CLKOUT3_BAR': '0', 'C_USE_CLKOUT4_BAR': '0', 'c_component_name': 'mts_MTSclkwiz_0', 'C_PLATFORM': 'UNKNOWN', 'C_USE_FREQ_SYNTH': '1', 'C_USE_PHASE_ALIGNMENT': '0', 'C_PRIM_IN_JITTER': '0.010', 'C_SECONDARY_IN_JITTER': '0.010', 'C_JITTER_SEL': 'Min_O_Jitt

## Basic Controls

In [14]:
def gpio_force_ch1_bit0_output(axigpio):
    mmio = axigpio.mmio
    tri = mmio.read(0x04)
    mmio.write(0x04, tri & ~0x1)

def gpio_set_ch1_bit0(axigpio, value):
    mmio = axigpio.mmio
    data = mmio.read(0x00)
    if value:
        mmio.write(0x00, data | 0x1)
    else:
        mmio.write(0x00, data & ~0x1)

def streamer_enable(on=True):
    g = ol.gpio_control.axi_gpio_dac
    gpio_force_ch1_bit0_output(g)
    gpio_set_ch1_bit0(g, 1 if on else 0)
    print('streamer_enable:', bool(on))

def digital_mute():
    ol.dac_player[:] = np.int16(0)
    streamer_enable(True)

streamer_enable(True)

streamer_enable: True


## MTS / RFDC Sanity

In [5]:
def find_rfdc(overlay):
    for name in dir(overlay):
        obj = getattr(overlay, name)
        if hasattr(obj, 'IPStatus') and ('rfdc' in name.lower() or 'rf_data_converter' in name.lower()):
            return obj
    if hasattr(overlay, 'xrfdc'):
        return overlay.xrfdc
    raise RuntimeError('RFDC object not found on overlay')

def read_clock_wizard_lock():
    try:
        lock = ol.clocktreeMTS.MTSclkwiz.read(0x0004)
        print('MTS clock wizard lock register:', hex(lock))
        return lock
    except Exception as e:
        print('Clock wizard lock read failed:', e)
        return None

def run_mts_sync():
    for method in ('init_tile_sync', 'verify_clock_tree', 'sync_tiles'):
        if hasattr(ol, method):
            status(method)
            try:
                getattr(ol, method)()
            except Exception as e:
                print(method, 'failed:', e)

def print_rfdc_summary():
    rfdc = find_rfdc(ol)
    print('IPStatus:', rfdc.IPStatus)
    for kind, tiles in [('DAC', rfdc.dac_tiles), ('ADC', rfdc.adc_tiles)]:
        for tile_id, tile in enumerate(tiles):
            try:
                print(f'{kind} tile {tile_id}: PLLLockStatus={tile.PLLLockStatus}, FIFOStatus={tile.FIFOStatus}')
            except Exception as e:
                print(f'{kind} tile {tile_id}: tile status read failed:', e)
            for block_id, block in enumerate(tile.blocks):
                try:
                    st = block.BlockStatus
                    print(f'  block {block_id}: SamplingFreq={st.get("SamplingFreq")}, DigitalPathEnabled={st.get("DigitalPathEnabled")}, DataPathClocksStatus={st.get("DataPathClocksStatus")}')
                except Exception as e:
                    print(f'  block {block_id}: block status read failed:', e)

read_clock_wizard_lock()
run_mts_sync()
print_rfdc_summary()

MTS clock wizard lock register: 0x1
[15:03:45] init_tile_sync
init_tile_sync failed: Function XRFdc_MultiConverter_Sync call failed
stdout: metal: error:     DAC tile 0 is not enabled for MTS, check IP configuration

[15:03:45] verify_clock_tree
[15:03:45] sync_tiles
sync_tiles failed: Function XRFdc_MultiConverter_Sync call failed
stdout: metal: error:     DAC tile 0 is not enabled for MTS, check IP configuration
metal: error:     DAC tile 2 in Multi-Tile group not started
metal: error:     DAC tile 2 is not enabled for MTS, check IP configuration
metal: error:     DAC2 block0 is not enabled, check IP configuration

IPStatus: {'DACTileStatus': [{'IsEnabled': 1, 'TileState': 15, 'BlockStatusMask': 1, 'PowerUpState': 1, 'PLLState': 1}, {'IsEnabled': 0, 'TileState': 0, 'BlockStatusMask': 0, 'PowerUpState': 0, 'PLLState': 0}, {'IsEnabled': 0, 'TileState': 0, 'BlockStatusMask': 0, 'PowerUpState': 0, 'PLLState': 0}, {'IsEnabled': 0, 'TileState': 0, 'BlockStatusMask': 0, 'PowerUpState': 0, '

## BRAM Integrity Test

Run this before enabling the DAC stream. If this fails, waveform tests are not meaningful.

In [6]:
def _as_i16_pattern(pattern, n):
    if pattern == 'zeros':
        return np.zeros(n, dtype=np.int16)
    if pattern == 'ones':
        return np.full(n, 0x7fff, dtype=np.int16)
    if pattern == 'alternating':
        x = np.empty(n, dtype=np.int16)
        x[0::2] = np.int16(0x5555)
        x[1::2] = np.int16(-0x5556)  # 0xAAAA as signed int16
        return x
    if pattern == 'ramp':
        return (np.arange(n, dtype=np.int32) & 0xffff).astype(np.int16)
    if pattern == 'prbs':
        rng = np.random.default_rng(0x12288)
        return rng.integers(-32768, 32767, size=n, dtype=np.int16)
    raise ValueError(pattern)

def bram_array_test(n=None, patterns=('zeros', 'ones', 'alternating', 'ramp', 'prbs')):
    streamer_enable(False)
    n = BUF_LEN if n is None else min(int(n), BUF_LEN)
    failures = []
    for pattern in patterns:
        expected = _as_i16_pattern(pattern, n)
        ol.dac_player[:n] = expected
        time.sleep(0.02)
        got = np.array(ol.dac_player[:n], dtype=np.int16, copy=True)
        bad = np.flatnonzero(got != expected)
        if len(bad):
            i = int(bad[0])
            failures.append((pattern, len(bad), i, int(expected[i]), int(got[i])))
            print(f'FAIL {pattern}: mismatches={len(bad)}, first={i}, expected={int(expected[i])}, got={int(got[i])}')
        else:
            print(f'PASS {pattern}: {n} samples')
    if failures:
        raise AssertionError(f'BRAM array test failed: {failures}')
    return True

bram_array_test(n=min(BUF_LEN, 2**18))

streamer_enable: False
PASS zeros: 131072 samples
PASS ones: 131072 samples
PASS alternating: 131072 samples
PASS ramp: 131072 samples
PASS prbs: 131072 samples


True

## Optional Raw MMIO BRAM Test

This bypasses the `dac_player` numpy view and writes through the AXI BRAM controller address range directly.

In [7]:
def raw_bram_mmio_test(ip='hier_dac_play/axi_bram_ctrl_0', words=4096):
    streamer_enable(False)
    info = ol.mem_dict[ip]
    base = int(info['phys_addr'])
    addr_range = int(info['addr_range'])
    words = min(int(words), addr_range // 4)
    mmio = MMIO(base, addr_range)
    patterns = [0x00000000, 0xffffffff, 0x5555aaaa, 0xaaaa5555]
    for pat in patterns:
        for i in range(words):
            mmio.write(i * 4, (pat + i) & 0xffffffff)
        bad = []
        for i in range(words):
            exp = (pat + i) & 0xffffffff
            got = mmio.read(i * 4) & 0xffffffff
            if got != exp:
                bad.append((i, exp, got))
                if len(bad) >= 8:
                    break
        if bad:
            print(f'FAIL raw pattern {pat:#010x}:', bad[:3])
            raise AssertionError('raw MMIO BRAM test failed')
        print(f'PASS raw pattern {pat:#010x}: {words} words')
    return True

raw_bram_mmio_test(words=4096)

streamer_enable: False
PASS raw pattern 0x00000000: 4096 words
PASS raw pattern 0xffffffff: 4096 words
PASS raw pattern 0x5555aaaa: 4096 words
PASS raw pattern 0xaaaa5555: 4096 words


True

## Program One Coherent Tone

In [83]:
# --- Waveform generation + plotting ---
from scipy.signal import sawtooth
from scipy.signal import square
import numpy as np
import matplotlib.pyplot as plt
DAC_AMP = 2**15 - 1
t_dac = 1/DAC_SR*np.arange(BUF_LEN)
def make_sine(freq_hz: float, amp: float = 1 * DAC_AMP) -> np.ndarray:
    y = amp * np.sin(2*np.pi*freq_hz*t_dac)
    return np.int16(np.round(np.clip(y, -DAC_AMP, DAC_AMP)))

def make_cos(freq_hz: float, amp: float = 1 * DAC_AMP) -> np.ndarray:
    y = amp * np.cos(2*np.pi*freq_hz*t_dac)
    return np.int16(np.round(np.clip(y, -DAC_AMP, DAC_AMP)))

def make_ramp(amp: float = 1 * DAC_AMP) -> np.ndarray:
    # deterministic pattern for debug (helps spot repeats/drops)
    y = np.linspace(-amp, amp, BUF_LEN, endpoint=False)
    return np.int16(np.round(np.clip(y, -DAC_AMP, DAC_AMP)))

def make_saw(freq_hz: float, amp: float = 1 * DAC_AMP) -> np.ndarray:
    if sawtooth is None:
        raise RuntimeError("scipy is not available; sawtooth() requires scipy.signal")
    y = amp * sawtooth(2*np.pi*freq_hz*(t_dac))
    
 #   return np.int16(np.round(np.clip(y, -DAC_AMP, DAC_AMP)))
    return np.int16(y)

def make_square(freq_hz: float, amp: float = 1 * DAC_AMP) -> np.ndarray:
    if sawtooth is None:
        raise RuntimeError("scipy is not available; square() requires scipy.signal")
    y = amp * square(2*np.pi*freq_hz*(t_dac))
    
 #   return np.int16(np.round(np.clip(y, -DAC_AMP, DAC_AMP)))
    return np.int16(y)

def make_coherent_sine(freq_hz, amp=0.5 * DAC_AMP):
    cycles = max(1, int(round(freq_hz * BUF_LEN / DAC_SR)))
    actual_freq = cycles * DAC_SR / BUF_LEN
    n = np.arange(BUF_LEN)
    y = amp * np.sin(2 * np.pi * cycles * n / BUF_LEN)
    y = np.clip(np.round(y), -32768, 32767).astype(np.int16)
    return y, actual_freq, cycles

def quick_fft(x, fs, n=2**16):
    x = np.asarray(x[:min(len(x), n)], dtype=float)
    x = x - np.mean(x)
    win = np.hanning(len(x))
    spec = np.fft.rfft(x * win)
    freq = np.fft.rfftfreq(len(x), 1 / fs)
    mag = 20 * np.log10(np.maximum(np.abs(spec), 1e-12))
    return freq, mag

def program_tone(freq_hz=50e6, amp=0.5 * DAC_AMP, enable=True):
    streamer_enable(False)
    sig, actual_freq, cycles = make_coherent_sine(freq_hz, amp=amp)
    ol.dac_player[:] = sig
    rb = np.array(ol.dac_player[:len(sig)], dtype=np.int16, copy=True)
    if not np.array_equal(rb, sig):
        bad = np.flatnonzero(rb != sig)
        i = int(bad[0])
        raise AssertionError(f'BRAM readback mismatch after tone write at {i}: expected={int(sig[i])}, got={int(rb[i])}')
    print(f'Requested {freq_hz/1e6:.6f} MHz, coherent {actual_freq/1e6:.6f} MHz, cycles={cycles}')
    f, mag = quick_fft(rb, DAC_SR)
    plt.figure(figsize=(9, 3))
    plt.plot(f / 1e6, mag)
    plt.xlim(0, min(500, DAC_SR / 2 / 1e6))
    plt.xlabel('Frequency (MHz)')
    plt.ylabel('Magnitude (dB)')
    plt.grid(True)
    plt.show()
    streamer_enable(enable)
    return sig, actual_freq

# signal, actual_freq = program_tone(freq_hz=1000e6, amp=0.7 * DAC_AMP, enable=True)
signal = make_saw(freq_hz=9830.4e6/9+1, amp=0.0 * DAC_AMP)
ol.dac_player[:] = signal

## Stop Output

In [ ]:
digital_mute()
# streamer_enable(False)